# Get To Know A Dataset: Open Subsurface Data

This notebook is a guided tour of the [Open Subsurface Data](https://registry.opendata.aws/open-subsurface-data/) dataset (the registry page goes live at launch). More usage examples, tutorials and documentation for this dataset and others are on the [Registry of Open Data on AWS](https://registry.opendata.aws/).

The dataset holds open 3D post-stack seismic volumes gathered from public archives, kept exactly as the sources shipped them, with the tables that say what each file is, how to read its headers and which Gulf of Mexico exploration wells sit inside each survey. Full documentation: [docs/dataset.md](https://github.com/mnky9800n/open-subsurface-data/blob/main/docs/dataset.md).

**Before launch.** The bucket does not exist yet. Set the environment variable `OSD_LOCAL_ROOT` to a folder laid out like the bucket and every cell reads from there instead. The same code runs against S3 once `BUCKET` is filled in.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The bucket follows the layout in the AWS onboarding handbook (Appendix A). Source files and the files we wrote never share a prefix.

```
s3://<bucket>/
  README.txt                        what the dataset is, licenses, how to cite
  data/
    original/                       the sources' files, byte for byte, unpacked from the source zips
      source=namss/survey=B-55d-99-LA/Data/G3D....sgy
      source=namss/survey=B-55d-99-LA/Navigation/...   (shapefiles, text, PDF reports as shipped)
      source=sarig/survey=mesac681/...
  metadata/
    index/                          the index tables: sources, files, volumes, headers, grids, wells
    surveys/source=<source>/survey=<survey_id>/metadata.json   header layout and fixes per volume
    releases/0.1/files.csv          every object in release 0.1: key, bytes, CRC32, SHA-256, source URL, license
  docs/                             this documentation and notebook
```

`source=` and `survey=` are Hive-style partition keys, so a tool that understands them (pandas, pyarrow, Amazon Athena) can filter on them. Inside a survey folder the paths are the ones inside the source's zip, so a survey looks the same here as it does after you unpack the original download. Nothing is zipped: you can read one volume without fetching the rest of the survey.

Release 0.1 has three parts: every SARIG 3D survey, every NAMSS survey offshore California and in the Beaufort Sea, and 20 Gulf of Mexico surveys (depth-migrated, clean headers, the most wells drilled after the survey was shot). The code below lists the bucket without an AWS account. From a shell the same thing is `aws s3 ls --no-sign-request s3://<bucket>/`.

In [ ]:
# This notebook needs (requirements.txt format):
# fsspec>=2024.6
# s3fs>=2024.6
# pandas>=2.2
# numpy>=1.26
# segyio>=1.9
# matplotlib>=3.8
#
# With uv: uv run --with fsspec --with s3fs --with pandas --with segyio --with matplotlib jupyter lab

First we import the libraries used in the rest of the notebook.

In [ ]:
import hashlib
import json
import os
import tempfile
from pathlib import Path

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segyio

Next we say where the data are. With `OSD_LOCAL_ROOT` set, `fs` is the local file system and `ROOT` a folder; without it, `fs` is anonymous S3 (no account, no keys) and `ROOT` the bucket. Everything after this cell uses only `fs` and `ROOT`.

In [ ]:
BUCKET = "<BUCKET-NAME>"   # pending: set when the bucket exists
LOCAL_ROOT = os.environ.get("OSD_LOCAL_ROOT")

if LOCAL_ROOT:
    fs = fsspec.filesystem("file")
    ROOT = str(Path(LOCAL_ROOT).expanduser().resolve())
else:
    fs = fsspec.filesystem("s3", anon=True)
    ROOT = BUCKET


def ls(prefix=""):
    """Names directly under a prefix, like `aws s3 ls`."""
    path = f"{ROOT}/{prefix}".rstrip("/")
    return sorted(p.rstrip("/").split("/")[-1] for p in fs.ls(path, detail=False))


print("reading from", ROOT if LOCAL_ROOT else f"s3://{BUCKET}")
print(ls())

The source files sit under `data/original/`, one prefix per source and then one per survey.

In [ ]:
print(ls("data/original"))
print(ls("data/original/source=namss")[:10])

A survey folder holds the SEG-Y volume and the side files that came in the same zip.

In [ ]:
survey_prefix = "data/original/source=namss/" + ls("data/original/source=namss")[0]
for p in fs.find(f"{ROOT}/{survey_prefix}"):
    print(p.removeprefix(ROOT + "/"), fs.size(p))

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

**SEG-Y** (`.sgy`, `.segy`) holds the seismic. A SEG-Y file has a 3,200-byte text header, a 400-byte binary header, and then one trace after another, each with a 240-byte trace header. Most files here are revision 0 or 1 with IBM floating-point samples. The standard puts the inline and crossline numbers in trace-header bytes 189 and 193, but most of these files put them somewhere else, and many text headers name the wrong bytes. Coordinates come in different bytes with different scalars, and the coordinate system is named in free text, if at all. Read SEG-Y with [segyio](https://github.com/equinor/segyio) (used below) and tell it where the inline and crossline numbers are.

**metadata.json** (one per survey, under `metadata/surveys/`) is what we found when we unpacked each volume: which header bytes hold inline, crossline and coordinates, the ranges and sample counts, the text header, whether segyio opens the file with full geometry, and every way the file departs from the standard. Use it to open a file correctly the first time.

**CSV and GeoJSON** (under `metadata/index/`) are the index tables. `sources.csv` gives each source's license and credit line. `seismic/volumes.csv` has one row per volume with region, year, stack type and domain. `wells/gulf_exploration_wells.csv` has every Gulf exploration well with its outcome and the surveys over it. `seismic/footprints.geojson` has survey outlines. The columns are described in `metadata/index/README.md`.

**Side files** are whatever the source put in the zip: PDF processing reports, ESRI shapefiles of the navigation, text notes and images. They are kept because they often hold the processing history and the coordinate system.

We keep the files exactly as the sources shipped them, so anyone can check a file against the source archive. That makes the SEG-Y harder to read than a cloud-native format would be, and the metadata is how we make up for it. If you compute on AWS, run in the bucket's region and read straight from S3 with `s3fs` or `boto3`; nothing needs to be copied first.

### Q: Can you show us an example of downloading and loading data from your dataset?

We start with the tables. The release file list has one row per object, with its region, its kind (`segy`, `side`, `metadata` or `index`) and its checksums.

In [ ]:
def read_csv(key, **kw):
    with fs.open(f"{ROOT}/{key}") as f:
        return pd.read_csv(f, **kw)


release = read_csv("metadata/releases/0.1/files.csv")
volumes = read_csv("metadata/index/seismic/volumes.csv")
sources = read_csv("metadata/index/sources.csv").set_index("source_id")

release.groupby(["part", "kind"]).agg(files=("s3_key", "size"), GB=("bytes", lambda b: round(b.sum() / 1e9, 1)))

Selecting by region and kind is a filter on those columns. Here are the SEG-Y volumes from the Gulf of Mexico, largest first.

In [ ]:
gulf_segy = release[(release.region == "gulf-of-mexico") & (release.kind == "segy")]
print(len(gulf_segy), "SEG-Y files,", round(gulf_segy.bytes.sum() / 1e9, 1), "GB")
gulf_segy.sort_values("bytes", ascending=False)[["survey_id", "s3_key", "bytes"]].head()

The index has more about each volume. For example, how the volumes offshore California and in South Australia split by domain (time, depth, or not stated by the source):

In [ ]:
in_release = set(release.survey_id.dropna())
(volumes[volumes.survey_id.isin(in_release) & volumes.region.isin(["pacific", "south-australia"])]
 .groupby(["region", "domain"], dropna=False).size())

Each source keeps its own license. Credit the sources you use with the line in `sources.csv`.

In [ ]:
for sid in ["namss", "sarig"]:
    print(sid, "|", sources.loc[sid, "license"])
    print("   credit:", sources.loc[sid, "attribution"], "\n")

Now one volume. We read the survey's `metadata.json` first, because it says which header bytes to use. The default below is one of the smallest clean volumes in release 0.1: the Unocal Hueneme survey offshore southern California (NAMSS B-430-85-SC, 279 MB), which segyio opens with full geometry once it knows the header bytes. Set `OSD_SURVEY` to `source/survey_id` to pick another.

In [ ]:
SOURCE, SURVEY = os.environ.get("OSD_SURVEY", "namss/B-430-85-SC").split("/")
meta = json.loads(fs.cat(f"{ROOT}/metadata/surveys/source={SOURCE}/survey={SURVEY}/metadata.json"))
name, vol = next(iter(meta["volumes"].items()))
hb = vol["header_bytes"]
print("volume:", name)
print("manifest:", vol["manifest"])
print("opens with segyio:", vol["opened_with_segyio"])
print("inline byte", hb["inline"], "| crossline byte", hb["crossline"], "| x/y bytes", hb["coordinates"]["x_byte"], hb["coordinates"]["y_byte"])
print("problems found:")
for p in vol["problems"]:
    print("  -", p)

segyio reads from a local path, so we download the file once and check it against the SHA-256 in the metadata. In the bucket's region this takes seconds.

In [ ]:
entry = vol["files"][0]
key = f"{ROOT}/data/original/source={SOURCE}/survey={SURVEY}/{entry['path']}"
local = Path(tempfile.gettempdir()) / "osd" / SURVEY / Path(entry["path"]).name
local.parent.mkdir(parents=True, exist_ok=True)
if not local.exists():
    fs.get(key, str(local))

h = hashlib.sha256()
with open(local, "rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        h.update(block)
print(local, local.stat().st_size, "bytes")
print("SHA-256 matches the metadata:", h.hexdigest() == entry["sha256"])

Now we open it with the inline and crossline bytes from the metadata, print the start of the text header and read the middle inline.

In [ ]:
with segyio.open(str(local), iline=hb["inline"], xline=hb["crossline"]) as f:
    text = f.text[0].decode("ascii", "replace")
    print("\n".join(text[i:i + 80] for i in range(0, 800, 80)))
    print()
    print("inlines", f.ilines[0], "to", f.ilines[-1], "| crosslines", f.xlines[0], "to", f.xlines[-1],
          "| samples", len(f.samples), "| interval", f.bin[segyio.BinField.Interval])
    il = f.ilines[len(f.ilines) // 2]
    section = f.iline[il].T          # samples down, crosslines across
    xlines, samples = f.xlines.copy(), f.samples.copy()
section.shape

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

First the inline we just read. The color scale is clipped at the 99th percentile of the amplitudes. The vertical axis is the sample value as stored; for NAMSS depth volumes that is the depth step times 1000 (see the note in the metadata).

In [ ]:
clip = np.percentile(np.abs(section), 99)
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(section, cmap="gray_r", vmin=-clip, vmax=clip, aspect="auto",
          extent=[xlines[0], xlines[-1], samples[-1], samples[0]])
ax.set_xlabel("crossline")
ax.set_ylabel("sample (as stored)")
ax.set_title(f"{SURVEY} {name}, inline {il}")
plt.show()

Next, where the Gulf part of the release sits. Gray lines are every NAMSS 3D survey outline in the index, blue lines the 20 Gulf surveys in release 0.1, and the dots are exploration wells drilled inside those surveys after they were shot (orange: the well discovered at least one sand in BOEM's atlas).

In [ ]:
footprints = json.loads(fs.cat(f"{ROOT}/metadata/index/seismic/footprints.geojson"))
wells = read_csv("metadata/index/wells/gulf_exploration_wells.csv", dtype={"api": str}, low_memory=False)
for c in ["bot_lon", "bot_lat", "surf_lon", "surf_lat"]:
    wells[c] = pd.to_numeric(wells[c], errors="coerce")
wells["lon"] = wells.bot_lon.fillna(wells.surf_lon)
wells["lat"] = wells.bot_lat.fillna(wells.surf_lat)

gulf_release = set(release[release.part == "namss-gulf-psdm"].survey_id)
year = volumes.drop_duplicates("survey_id").set_index("survey_id").year
pairs = wells.assign(survey=wells.surveys.str.split(";")).explode("survey")
pairs = pairs[pairs.survey.isin(gulf_release) & (pairs.spud_year > pairs.survey.map(year))]

fig, ax = plt.subplots(figsize=(12, 7))
for feat in footprints["features"]:
    p = feat["properties"]
    if p["source"] != "namss" or p["footprint"] != "NAMSS navigation outline":
        continue
    rings = feat["geometry"]["coordinates"]
    rings = rings if feat["geometry"]["type"] == "Polygon" else [r for poly in rings for r in poly]
    chosen = p["survey_id"] in gulf_release
    for ring in rings:
        xy = np.array(ring)
        ax.plot(xy[:, 0], xy[:, 1], color="tab:blue" if chosen else "0.75", lw=1.2 if chosen else 0.4)
w = pairs.drop_duplicates("api")
ax.scatter(w.lon, w.lat, s=4, color=np.where(w.discovery, "tab:orange", "0.2"), zorder=3)
ax.set_xlim(-98, -86); ax.set_ylim(26, 30.5); ax.set_aspect(1 / np.cos(np.radians(28)))
ax.set_xlabel("longitude (NAD27)"); ax.set_ylabel("latitude (NAD27)")
ax.set_title(f"{len(gulf_release)} Gulf surveys in release 0.1 and {len(w)} wells drilled inside them after the survey")
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

How often did an exploration well drilled inside a 3D survey, after the survey was shot, find pay? The wells table answers it for the whole Gulf. A well counts as a discovery when BOEM's sands atlas names it as the discovery well of at least one sand. That makes the rate a lower bound: a well that found pay in a sand someone else had already found counts as a miss.

In [ ]:
after = wells[wells.after_survey == True]
summary = after.groupby("setting").discovery.agg(wells="size", discoveries="sum")
summary["rate"] = (summary.discoveries / summary.wells).round(3)
summary.loc["all"] = [len(after), after.discovery.sum(), round(after.discovery.mean(), 3)]
summary

The same count for each Gulf survey in the release, which is how those surveys were chosen (the most wells drilled after the shoot):

In [ ]:
per = pairs.groupby("survey").agg(wells=("api", "nunique"), discoveries=("discovery", "sum"))
per = per.sort_values("wells")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(per.index, per.wells, color="0.6", label="wells drilled after the survey")
ax.barh(per.index, per.discoveries, color="tab:orange", label="discovery wells")
ax.set_xlabel("wells inside the survey outline")
ax.legend(loc="lower right")
plt.show()

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

Can the seismic around a well, as it looked before the well was drilled, predict whether the well found pay?

The pieces are here. The wells table gives the outcome and the spud date, every Gulf survey in the release was shot before the wells counted against it, and `metadata/index/wells/gulf_well_volume_ties.csv` places wells in the inline and crossline grid of the volumes whose coordinates we could fit. Some advice:

- Use only ties with `grid_quality` of `good`, and check the coordinate system in `seismic/grids.csv`.
- Split training and test data by field or by area. Wells in one field share their seismic and their outcome, so a random split leaks.
- Treat the labels as a lower bound, as above.
- The volumes are depth migrated, so a window around the well's bottom-hole position in inline, crossline and depth is a natural input. A pretrained seismic encoder is a good place to start.

If you try it, tell us: open an issue at https://github.com/mnky9800n/open-subsurface-data/issues.